## 1. Environment and PDF Library Verification

Before implementing PDF extraction, we verify the Python
environment and the availability of the required libraries.

PyMuPDF will be the primary PDF extraction library.

pypdf will be retained as a secondary library for future
comparison and alternative PDF operations.

In [1]:

# ============================================================
# Advance AI Research Assistant
# Notebook 03 – PDF Extraction
# Section 5.2 – Import Required Libraries
# ============================================================

import sys
from pathlib import Path

import fitz  # PyMuPDF
import pypdf

import pandas as pd
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [4]:
# Check the Python environment

print("Python version:", sys.version)
print("Python executable:", sys.executable)

# Check the project environment

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

print("\nProject root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())


Python version: 3.11.17 (main, Oct 13 2026, 16:45:47) [GCC 15.3.0]
Python executable: /home/r/miniconda3/envs/nlp/bin/python

Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Project root exists: True


In [3]:
# Verify the PDF libraries
# PyMuPDF version
print("PyMuPDF version:", fitz.VersionBind)

# pypdf version
print("pypdf version:", pypdf.__version__)

# Verify PyMuPDF functionality
print("\nPyMuPDF module:", fitz.__name__)
print("PyMuPDF available:", hasattr(fitz, "open"))

# Verify pypdf functionality
print("pypdf available:", hasattr(pypdf, "PdfReader"))


PyMuPDF version: 1.26.7
pypdf version: 6.21.2

PyMuPDF module: fitz
PyMuPDF available: True
pypdf available: True


In [5]:
# Verify the document directory
# Define the document directories

DATA_DIR = PROJECT_ROOT / "data"

RAW_PDF_DIR = DATA_DIR / "raw_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"

# Create directories if they do not exist

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw PDF directory:", RAW_PDF_DIR)
print("Processed directory:", PROCESSED_DIR)

print("\nDirectories ready.")


Raw PDF directory: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/raw_pdfs
Processed directory: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed

Directories ready.


## 2. PDF Document Discovery

In this section, we identify PDF documents available in the
raw document directory.

The objectives are:

- Find all PDF files.
- Inspect their filenames and file sizes.
- Identify empty or unusually small files.
- Prepare a list of documents for extraction.

Original PDF files must remain unchanged.

In [7]:

# ============================================================
# 2.2 Discover PDF Documents
# ============================================================

# Find all PDF files recursively
pdf_files = sorted(RAW_PDF_DIR.rglob("*.pdf"))

print(f"Number of PDF files found: {len(pdf_files)}")

# Display discovered documents
for pdf_file in pdf_files:
    print(pdf_file.name)


Number of PDF files found: 3
41598_2019_Article_38798.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
s41598-019-52937-5.pdf


In [8]:
# ============================================================
# 2.3 Inspect PDF File Sizes
# ============================================================

pdf_inventory = []

for pdf_file in pdf_files:

    file_size_bytes = pdf_file.stat().st_size
    file_size_mb = file_size_bytes / (1024 * 1024)

    pdf_inventory.append({
        "filename": pdf_file.name,
        "file_path": str(pdf_file),
        "size_bytes": file_size_bytes,
        "size_mb": round(file_size_mb, 3),
        "is_empty": file_size_bytes == 0
    })

# Convert results into a DataFrame
pdf_inventory_df = pd.DataFrame(pdf_inventory)

display(pdf_inventory_df)


,filename,file_path,size_bytes,size_mb,is_empty
0,41598_2019_Article_38798.pdf,/home/r/Schreibtisch/IfaDW/NLP/Final_Project/a...,11424792,10.896,False
1,NIPS-2017-attention-is-all-you-need-Paper.pdf,/home/r/Schreibtisch/IfaDW/NLP/Final_Project/a...,569417,0.543,False
2,s41598-019-52937-5.pdf,/home/r/Schreibtisch/IfaDW/NLP/Final_Project/a...,3388120,3.231,False


In [9]:

# ============================================================
# 2.4 Basic Document Availability Check
# ============================================================

if not pdf_files:
    print("No PDF documents found.")
    print("Please add at least one PDF to:", RAW_PDF_DIR)

else:
    print(f"{len(pdf_files)} PDF document(s) available.")

    empty_files = [
        path for path in pdf_files
        if path.stat().st_size == 0
    ]

    print(f"Empty files: {len(empty_files)}")

    if empty_files:
        print("\nEmpty PDF files:")
        for path in empty_files:
            print("-", path.name)


3 PDF document(s) available.
Empty files: 0


## 3. First PDF Extraction with PyMuPDF

In this section, we perform our first PDF text extraction
using PyMuPDF.

We will:

- Select a scientific research paper.
- Open the PDF document.
- Inspect its basic properties.
- Extract text from individual pages.
- Display a small text sample.
- Verify that the extracted text is meaningful.

PyMuPDF is the primary extraction library.

In [10]:

# ============================================================
# 3.2 Select a Test PDF
# ============================================================

# Select the Attention Is All You Need research paper
TEST_PDF = RAW_PDF_DIR / "NIPS-2017-attention-is-all-you-need-Paper.pdf"

# Verify that the document exists
if not TEST_PDF.exists():
    raise FileNotFoundError(f"PDF not found: {TEST_PDF}")

print("Selected PDF:", TEST_PDF.name)
print("File size (MB):", round(TEST_PDF.stat().st_size / (1024 * 1024), 3))


Selected PDF: NIPS-2017-attention-is-all-you-need-Paper.pdf
File size (MB): 0.543


In [11]:

# ============================================================
# 3.3 Open and Inspect PDF
# ============================================================

# Open the PDF using PyMuPDF
with fitz.open(TEST_PDF) as pdf_document:

    print("PDF opened successfully.")

    print("\nDocument information:")
    print("Number of pages:", pdf_document.page_count)
    print("PDF format:", pdf_document.metadata.get("format"))
    print("Title:", pdf_document.metadata.get("title"))
    print("Author:", pdf_document.metadata.get("author"))
    print("Subject:", pdf_document.metadata.get("subject"))
    print("Creator:", pdf_document.metadata.get("creator"))
    print("Encrypted:", pdf_document.is_encrypted)


PDF opened successfully.

Document information:
Number of pages: 11
PDF format: PDF 1.3
Title: Attention is All you Need
Author: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin
Subject: Neural Information Processing Systems http://nips.cc/
Creator: 
Encrypted: False


In [12]:
# ============================================================
# 3.4 Extract Text from the First Page
# ============================================================

with fitz.open(TEST_PDF) as pdf_document:

    # Get the first page (Python indexing starts at 0)
    first_page = pdf_document[0]

    # Extract plain text
    first_page_text = first_page.get_text("text")

    print("First page extracted successfully.")
    print("Extracted characters:", len(first_page_text))

    print("\nFirst 2000 characters:\n")
    print(first_page_text[:2000])


First page extracted successfully.
Extracted characters: 2908

First 2000 characters:

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality whi

In [13]:

# ============================================================
# 3.5 Extract Text from All Pages
# ============================================================

extracted_pages = []

with fitz.open(TEST_PDF) as pdf_document:

    for page_index, page in enumerate(pdf_document):

        # Extract text from the current page
        page_text = page.get_text("text")

        extracted_pages.append({
            "page_number": page_index + 1,
            "text": page_text,
            "character_count": len(page_text)
        })

print("Total pages processed:", len(extracted_pages))

total_characters = sum(
    page["character_count"] for page in extracted_pages
)

print("Total extracted characters:", total_characters)

# Display extraction statistics
extraction_summary = pd.DataFrame([
    {
        "page_number": page["page_number"],
        "character_count": page["character_count"]
    }
    for page in extracted_pages
])

display(extraction_summary)


Total pages processed: 11
Total extracted characters: 32709


,page_number,character_count
0,1,2908
1,2,4254
2,3,1756
3,4,2449
4,5,3207
5,6,3545
6,7,3228
7,8,3328
8,9,2618
9,10,3090


## 4. PDF Metadata and Page-Level Extraction

This section develops a structured representation of extracted
PDF content.

The objectives are:

- Extract complete PDF metadata.
- Preserve page-level text.
- Record page dimensions.
- Record character counts.
- Detect pages without extractable text.
- Maintain traceability to the original PDF.

This structured representation will later support document
chunking, vector storage, and page-level citations.

In [14]:

# ============================================================
# 4.2 Extract PDF Metadata
# ============================================================

with fitz.open(TEST_PDF) as pdf_document:

    pdf_metadata = pdf_document.metadata

    metadata = {
        "filename": TEST_PDF.name,
        "file_path": str(TEST_PDF),
        "file_size_bytes": TEST_PDF.stat().st_size,
        "page_count": pdf_document.page_count,
        "pdf_version": pdf_metadata.get("format"),
        "title": pdf_metadata.get("title"),
        "author": pdf_metadata.get("author"),
        "subject": pdf_metadata.get("subject"),
        "keywords": pdf_metadata.get("keywords"),
        "creator": pdf_metadata.get("creator"),
        "producer": pdf_metadata.get("producer"),
        "creation_date": pdf_metadata.get("creationDate"),
        "modification_date": pdf_metadata.get("modDate"),
        "is_encrypted": pdf_document.is_encrypted
    }

metadata_df = pd.DataFrame(
    list(metadata.items()),
    columns=["Metadata Field", "Value"]
)

display(metadata_df)


,Metadata Field,Value
0,filename,NIPS-2017-attention-is-all-you-need-Paper.pdf
1,file_path,/home/r/Schreibtisch/IfaDW/NLP/Final_Project/a...
2,file_size_bytes,569417
3,page_count,11
4,pdf_version,PDF 1.3
5,title,Attention is All you Need
6,author,"Ashish Vaswani, Noam Shazeer, Niki Parmar, Jak..."
7,subject,Neural Information Processing Systems http://n...
8,keywords,
9,creator,


In [15]:

# ============================================================
# 4.3 Structured Page-Level Extraction
# ============================================================

structured_pages = []

with fitz.open(TEST_PDF) as pdf_document:

    for page_index, page in enumerate(pdf_document):

        page_text = page.get_text("text")

        structured_pages.append({
            "document_filename": TEST_PDF.name,
            "page_number": page_index + 1,
            "text": page_text,
            "character_count": len(page_text),
            "word_count": len(page_text.split()),
            "page_width": round(page.rect.width, 2),
            "page_height": round(page.rect.height, 2),
            "is_empty": len(page_text.strip()) == 0
        })

structured_pages_df = pd.DataFrame(structured_pages)

display(structured_pages_df.drop(columns=["text"]))

print("Total pages:", len(structured_pages))
print(
    "Total words:",
    structured_pages_df["word_count"].sum()
)
print(
    "Empty pages:",
    structured_pages_df["is_empty"].sum()
)


,document_filename,page_number,character_count,word_count,page_width,page_height,is_empty
0,NIPS-2017-attention-is-all-you-need-Paper.pdf,1,2908,400,612.0,792.0,False
1,NIPS-2017-attention-is-all-you-need-Paper.pdf,2,4254,629,612.0,792.0,False
2,NIPS-2017-attention-is-all-you-need-Paper.pdf,3,1756,282,612.0,792.0,False
3,NIPS-2017-attention-is-all-you-need-Paper.pdf,4,2449,384,612.0,792.0,False
4,NIPS-2017-attention-is-all-you-need-Paper.pdf,5,3207,531,612.0,792.0,False
5,NIPS-2017-attention-is-all-you-need-Paper.pdf,6,3545,547,612.0,792.0,False
6,NIPS-2017-attention-is-all-you-need-Paper.pdf,7,3228,520,612.0,792.0,False
7,NIPS-2017-attention-is-all-you-need-Paper.pdf,8,3328,553,612.0,792.0,False
8,NIPS-2017-attention-is-all-you-need-Paper.pdf,9,2618,428,612.0,792.0,False
9,NIPS-2017-attention-is-all-you-need-Paper.pdf,10,3090,411,612.0,792.0,False


Total pages: 11
Total words: 5001
Empty pages: 0


In [16]:

# ============================================================
# 4.4 Inspect a Selected Page
# ============================================================

# Select page 2 of the PDF
selected_page_number = 2

selected_page = next(
    page for page in structured_pages
    if page["page_number"] == selected_page_number
)

print(f"Filename: {selected_page['document_filename']}")
print(f"Page: {selected_page['page_number']}")
print(f"Characters: {selected_page['character_count']}")
print(f"Words: {selected_page['word_count']}")

print("\nExtracted text:\n")
print(selected_page["text"])


Filename: NIPS-2017-attention-is-all-you-need-Paper.pdf
Page: 2
Characters: 4254
Words: 629

Extracted text:

Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
signiﬁcant improvements in computational efﬁciency through factorization tricks [18] and conditional
computation [26], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing m

### 5. Reading Order: Default vs. Sorted Extraction

PyMuPDF provides different ways to extract text from a PDF page.

- Default extraction follows the PDF's internal content order.
- Sorted extraction attempts to arrange text according to its position on the page.

We compare both methods to understand their effect on reading order.

In [17]:

# ============================================================
# 5.1 Reading Order
# ============================================================
# Select page 2 for our reading-order experiment.
# PyMuPDF uses zero-based page indexing.
PAGE_INDEX = 1

with fitz.open(TEST_PDF) as pdf_document:
    page = pdf_document[PAGE_INDEX]

    # Default extraction
    text_default = page.get_text("text")

    # Extraction with spatial sorting
    text_sorted = page.get_text("text", sort=True)

print("DEFAULT EXTRACTION")
print("=" * 80)
print(text_default[:2500])

print("\n\nSORTED EXTRACTION")
print("=" * 80)
print(text_sorted[:2500])

DEFAULT EXTRACTION
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
signiﬁcant improvements in computational efﬁciency through factorization tricks [18] and conditional
computation [26], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [

### 5.2 Text Blocks and Bounding Boxes

We extract individual text blocks and their coordinates.

This information helps us understand the spatial organization of scientific documents and provides a foundation for future layout-aware processing.

In [18]:
#===========================================================================
# 5.2 Text Blocks and Bounding Boxes
#===========================================================================

blocks_data = []

with fitz.open(TEST_PDF) as pdf_document:
    page = pdf_document[PAGE_INDEX]

    blocks = page.get_text("blocks")

    for block_index, block in enumerate(blocks):

        x0, y0, x1, y1, text, block_number, block_type = block[:7]

        blocks_data.append({
            "block_index": block_index,
            "block_number": block_number,
            "block_type": block_type,
            "x0": round(x0, 2),
            "y0": round(y0, 2),
            "x1": round(x1, 2),
            "y1": round(y1, 2),
            "width": round(x1 - x0, 2),
            "height": round(y1 - y0, 2),
            "character_count": len(text.strip()),
            "text_preview": text.strip().replace("\n", " ")[:120]
        })

blocks_df = pd.DataFrame(blocks_data)

display(blocks_df)

,block_index,block_number,block_type,x0,y0,x1,y1,width,height,character_count,text_preview
0,0,0,0,108.00,74.33,504.35,160.73,396.35,86.40,753,Recurrent models typically factor computation ...
1,1,1,0,107.64,167.18,505.66,209.85,398.01,42.67,348,Attention mechanisms have become an integral p...
2,2,2,0,107.69,216.20,505.74,258.97,398.05,42.76,382,"In this work we propose the Transformer, a mod..."
3,3,3,0,108.00,275.68,188.83,287.64,80.83,11.96,12,2 Background
4,4,4,0,107.69,300.37,505.24,397.56,397.55,97.19,824,The goal of reducing sequential computation al...
5,5,5,0,108.00,404.04,505.25,446.67,397.25,42.64,393,"Self-attention, sometimes called intra-attenti..."
6,6,6,0,108.00,453.02,505.65,484.88,397.65,31.86,219,End-to-end memory networks are based on a recu...
7,7,7,0,107.69,491.23,505.66,534.00,397.97,42.76,374,"To the best of our knowledge, however, the Tra..."
8,8,8,0,108.00,550.71,226.09,562.67,118.09,11.96,20,3 Model Architecture
9,9,9,0,108.00,575.35,505.74,628.95,397.74,53.60,481,Most competitive neural sequence transduction ...


In [19]:
#===========================================================================
# 5.3 Inspect word-level positions
#===========================================================================
# We can also extract individual words and their coordinates.
# This is useful when we eventually need more precise layout analysis.

with fitz.open(TEST_PDF) as pdf_document:
    page = pdf_document[PAGE_INDEX]

    words = page.get_text("words")

words_data = []

for word in words:
    x0, y0, x1, y1, word_text = word[:5]

    words_data.append({
        "word": word_text,
        "x0": round(x0, 2),
        "y0": round(y0, 2),
        "x1": round(x1, 2),
        "y1": round(y1, 2),
    })

words_df = pd.DataFrame(words_data)

display(words_df.head(30))

print("Total extracted words:", len(words_df))

,word,x0,y0,x1,y1
0,Recurrent,108.00,74.33,148.07,84.39
1,models,150.57,74.33,179.93,84.39
2,typically,182.44,74.33,218.00,84.39
3,factor,220.50,74.33,244.10,84.39
4,computation,246.61,74.33,297.42,84.39
5,along,299.92,74.33,322.50,84.39
6,the,325.00,74.33,337.41,84.39
7,symbol,339.92,74.33,369.85,84.39
8,positions,372.35,74.33,409.06,84.39
9,of,411.57,74.33,420.03,84.39


Total extracted words: 629


In [20]:
#===========================================================================
# 5.4 Compare the number of blocks and words
#===========================================================================

print("Page number:", PAGE_INDEX + 1)
print("Number of text blocks:", len(blocks_df))
print("Number of extracted words:", len(words_df))
print("Default text characters:", len(text_default))
print("Sorted text characters:", len(text_sorted))

Page number: 2
Number of text blocks: 14
Number of extracted words: 629
Default text characters: 4254
Sorted text characters: 4310
